In [19]:
from openai import OpenAI
from pydantic import BaseModel
import os
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY not found in environment variables")
client= OpenAI(api_key=api_key)

In [20]:

class InstructionResponseList(BaseModel):
    responses: list[str]

response = client.responses.parse(
    model="gpt-4o-mini",
    input=[{
        "role": "user", 
        "content": """You are asked to come up 2 questions on Electrical Engineering based on the provided context.
Please follow these guiding principles when generating responses:
* Use proper grammar and punctuation.
* Always generate safe and respectful content. Do not generate content that is harmful, abusive, or offensive.
* Always generate content that is factually accurate and relevant to the prompt.
* The questions should be clear and human-like.
* The questions should be diverse and cover a wide range of topics.
* The questions should not be template-based or generic, it should be very diverse.
* Simply return the questions, do not return any answers or explanations.
* Strictly adhere to the prompt and generate responses in the same style and format as the example.

To better assist you with this task, here is an example:
### Question:
1. What are the key design considerations for a low-noise amplifier in RF circuits?

Context:A reliable single-line diagram of an industrial or commercial electrical power distribution\nsystem is an invaluable tool. It is also called a one-line diagram. The single-line diagram indicates, by single lines and standard symbols, the course and component parts of an electric circuit or system of circuits. The symbols that are commonly used in one-line diagrams are\ndefined in IEEE Std 315-1975. [1]\nThe single-line diagram is a road map of the distribution system that traces the flow of power\ninto and through the system. The single-line drawing identifies the points at which power is,\nor can be, supplied into the system and at which power should be disconnected in order to\nclear, or isolate, any portion of the system.

Now generate 2 such questions, remember to follow the principles mentioned above. No numbering needed.
and use the same format as the examples. Remember to use the same style and format as the example
above.

"""}],
    text_format=InstructionResponseList,
    temperature=0.7
)
print(response.output_parsed)

responses=['What role does a single-line diagram play in ensuring the safety and reliability of an electrical power distribution system?', 'How do standard symbols defined in IEEE Std 315-1975 enhance the clarity and effectiveness of single-line diagrams in electrical engineering?']


In [23]:
class EvaluationResponse(BaseModel):
    score: int
    explanation: str

parsed_response = client.responses.parse(
    model="gpt-4o-mini",
    input=[{"role": "user", "content": "Evaluate the following question: " + "".join(response.output_parsed.responses)}],
    temperature=0.7,
    text_format=EvaluationResponse
)
print(parsed_response.output_parsed)

score=8 explanation="A single-line diagram (SLD) is crucial in electrical power distribution systems as it provides a simplified representation of the system's layout, showing how components like transformers, circuit breakers, and loads are interconnected. This clarity helps engineers and technicians quickly understand the system's configuration, which is essential for troubleshooting, maintenance, and ensuring safety measures are in place. \n\nStandard symbols defined in IEEE Std 315-1975 enhance this clarity by providing a universally recognized set of symbols for electrical components. This standardization reduces ambiguity, allowing engineers from different backgrounds to interpret the diagrams consistently and accurately. As a result, it improves communication among stakeholders and aids in the training of personnel, ultimately reinforcing the reliability and safety of the electrical distribution system."


In [31]:
import os
import concurrent.futures
from openai import OpenAI
import time

# --- Configuration ---
if "OPENAI_API_KEY" not in os.environ:
    print("Error: OPENAI_API_KEY environment variable not set.")
    exit(1)

# 1. Initialize the REGULAR (synchronous) client
client = OpenAI()

# 2. Define the function that makes a single, regular API call
def get_completion(prompt: str) -> str | None:
    """Makes a single synchronous call to the OpenAI API."""
    print(f"[Thread] STARTING: {prompt}")
    try:
        completion = client.responses.create(
            model="gpt-5-mini",
            input=[
                {"role": "system", "content": "You are a helpful assistant."},
                {"role": "user", "content": prompt}
            ],
            service_tier="flex"
        )
        result = None
        if completion:
            try:
                if completion.output and len(completion.output) > 0:
                    output_item = completion.output[0]
                    content = getattr(output_item, 'content', None)
                    if content and len(content) > 0:
                        result = content[0].text
            except (AttributeError, IndexError, TypeError):
                pass
        print(f"[Thread] FINISHED: {prompt}")
        return result
    except Exception as e:
        print(f"[Thread] ERROR on '{prompt}': {e}")
        return None

# 3. Define the list of prompts you want to send
prompts = [
    "What are the common challenges faced in maintaining accurate equipment documentation for industrial power systems",
    "How can the implementation of computer-aided drafting systems improve the efficiency of updating electrical diagrams?",
    "What role does unique identification play in preventing operational errors within electrical systems?"
]

print("--- THREADED (CONCURRENT) TEST ---")
print(f"Starting {len(prompts)} concurrent API calls...")

# --- TIMER START ---
start_time = time.perf_counter()

# 4. Use ThreadPoolExecutor to run the function concurrently
results = []
with concurrent.futures.ThreadPoolExecutor(max_workers=len(prompts)) as executor:
    results = list(executor.map(get_completion, prompts))

# --- TIMER END ---
end_time = time.perf_counter()
total_time = end_time - start_time

print("\n--- All concurrent requests have completed ---")
print(f"TOTAL TIME (Threaded): {total_time:.2f} seconds")

# 5. Print the results
for i, (prompt, result) in enumerate(zip(prompts, results)):
    print(f"\nResult for Prompt {i+1} ('{prompt}'):")
    print(f"  -> {result}")

--- THREADED (CONCURRENT) TEST ---
Starting 3 concurrent API calls...
[Thread] STARTING: What are the common challenges faced in maintaining accurate equipment documentation for industrial power systems
[Thread] STARTING: How can the implementation of computer-aided drafting systems improve the efficiency of updating electrical diagrams?
[Thread] STARTING: What role does unique identification play in preventing operational errors within electrical systems?
[Thread] FINISHED: What are the common challenges faced in maintaining accurate equipment documentation for industrial power systems
[Thread] FINISHED: What role does unique identification play in preventing operational errors within electrical systems?
[Thread] FINISHED: How can the implementation of computer-aided drafting systems improve the efficiency of updating electrical diagrams?

--- All concurrent requests have completed ---
TOTAL TIME (Threaded): 29.24 seconds

Result for Prompt 1 ('What are the common challenges faced in 

In [29]:
import os
from openai import OpenAI
import time

# --- Configuration ---
if "OPENAI_API_KEY" not in os.environ:
    print("Error: OPENAI_API_KEY environment variable not set.")
    exit(1)

# 1. Initialize the REGULAR (synchronous) client
client = OpenAI()

# 2. Define the function that makes a single, regular API call
def get_completion(prompt: str) -> str | None:
    """Makes a single synchronous call to the OpenAI API."""
    print(f"[Main] STARTING: {prompt}")
    try:
        completion = client.responses.create(
            model="gpt-5-mini",
            input=[
                {"role": "system", "content": "You are a helpful assistant."},
                {"role": "user", "content": prompt}
            ],
            service_tier="flex"

        )
        result = None
        if completion:
            try:
                if completion.output and len(completion.output) > 0:
                    output_item = completion.output[0]
                    content = getattr(output_item, 'content', None)
                    if content and len(content) > 0:
                        result = content[0].text
            except (AttributeError, IndexError, TypeError):
                pass
        print(f"[Main] FINISHED: {prompt}")
        return result
    except Exception as e:
        print(f"[Main] ERROR on '{prompt}': {e}")
        return None

# 3. Define the list of prompts you want to send
prompts = [
    "What are the common challenges faced in maintaining accurate equipment documentation for industrial power systems",
    "How can the implementation of computer-aided drafting systems improve the efficiency of updating electrical diagrams?",
    "What role does unique identification play in preventing operational errors within electrical systems?"
]

print("--- SYNCHRONOUS (SEQUENTIAL) TEST ---")
print(f"Starting {len(prompts)} sequential API calls...")

# --- TIMER START ---
start_time = time.perf_counter()

# 4. Run the function sequentially in a simple loop
results = []
for prompt in prompts:
    # This will WAIT for the function to finish before starting the next one
    result = get_completion(prompt)
    results.append(result)

# --- TIMER END ---
end_time = time.perf_counter()
total_time = end_time - start_time

print("\n--- All sequential requests have completed ---")
print(f"TOTAL TIME (Synchronous): {total_time:.2f} seconds")

# 5. Print the results
for i, (prompt, result) in enumerate(zip(prompts, results)):
    print(f"\nResult for Prompt {i+1} ('{prompt}'):")
    print(f"  -> {result}")

--- SYNCHRONOUS (SEQUENTIAL) TEST ---
Starting 3 sequential API calls...
[Main] STARTING: What are the common challenges faced in maintaining accurate equipment documentation for industrial power systems
[Main] FINISHED: What are the common challenges faced in maintaining accurate equipment documentation for industrial power systems
[Main] STARTING: How can the implementation of computer-aided drafting systems improve the efficiency of updating electrical diagrams?
[Main] FINISHED: How can the implementation of computer-aided drafting systems improve the efficiency of updating electrical diagrams?
[Main] STARTING: What role does unique identification play in preventing operational errors within electrical systems?
[Main] FINISHED: What role does unique identification play in preventing operational errors within electrical systems?

--- All sequential requests have completed ---
TOTAL TIME (Synchronous): 34.98 seconds

Result for Prompt 1 ('What are the common challenges faced in mainta